# 08 — Migration Results

Consolidated view across every step: accounts, addresses, and orders.
Reads only from `migration_data/` — safe to re-run any time without
touching OneBill.

## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

df_account_results  = try_load_df("account_results")
df_address_results  = try_load_df("address_results")
df_order_results    = try_load_df("order_results")


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 27
python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 38
python-dotenv could not parse statement starting at line 44


## 2. Account creation summary

In [2]:
if df_account_results is not None:
    print(df_account_results["status"].value_counts().to_string())
    display(df_account_results)
else:
    print("04_Create_Accounts.ipynb has not been run yet.")


status
created    2


,AccountCode,AccountCode_Batch,AccountName,AccountKey,status,onebill_id,error
0,SR1404,SR1404,Managed by Williams (SR1404),managed_by_williams,created,unknown,NaN
1,SR1602,SR1602,Williams Internet Limited (SR1602),NaN,created,unknown,NaN


## 3. Address creation summary + failures

In [3]:
if df_address_results is not None:
    print(df_address_results["status"].value_counts().to_string())
    address_failures = df_address_results[df_address_results["status"] != "created"]
    print(f"\n{len(address_failures):,} not created (failed or skipped)")
else:
    address_failures = pd.DataFrame()
    print("06_Create_Addresses.ipynb has not been run yet.")

address_failures.head(20)



status
created    231
exists      25
skipped      8

33 not created (failed or skipped)


,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error
32,V113080527,SR1404,10/346 CASHEL STREET,exists,2104.0,NaN
33,V113080956,SR1404,102/171 ST ASAPH STREET,exists,1405.0,NaN
37,V113081020,SR1404,3/365 MADRAS STREET,exists,2215.0,NaN
57,V113068670,SR1602,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...
63,V113085682,SR1404,2/23 ALLEN STREET,exists,2213.0,NaN
65,V113086615,SR1404,3/245 MOORHOUSE AVENUE,exists,2208.0,NaN
74,V113079503,SR1404,G12/20 BATH STREET,exists,811.0,NaN
78,V113068308,SR1602,103/171 ST ASAPH STREET,exists,2108.0,NaN
79,V113083844,SR1404,6/43 PORUTU STREET,exists,2109.0,NaN
83,V113079289,SR1404,3/73 FERRY ROAD,exists,2217.0,NaN


In [4]:
no_address = df_address_results[df_address_results["status"].isin(["failed", "skipped"])].copy()

df_subs_lookup = try_load_df("subscriptions_resolved")
if df_subs_lookup is not None:
    no_address = no_address.merge(
        df_subs_lookup[["SubscriptionUSN", "SupplierServiceID"]],
        on="SubscriptionUSN",
        how="left",
    )

out_path = f'Migration_data/Subscriptions_Without_Address_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
no_address.to_csv(out_path, index=False)
print(f"Wrote {len(no_address):,} subscriptions without an address to {out_path}")
no_address.head(20)

Wrote 8 subscriptions without an address to Migration_data/Subscriptions_Without_Address_20260729_080503.csv


,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error,SupplierServiceID
0,V113068670,SR1602,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02624632
1,V113062335,SR1602,NaN,skipped,NaN,no SupplierServiceID on this subscription,NaN
2,V113066765,SR1404,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02622491
3,V113074579,SR1602,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,1643259206
4,V113071120,SR1404,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02627005
5,V113076806,SR1404,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02633693
6,V113071849,SR1602,NaN,skipped,NaN,no SupplierServiceID on this subscription,NaN
7,V113062343,SR1602,NaN,skipped,NaN,no SupplierServiceID on this subscription,NaN


In [5]:
if not address_failures.empty:
    out_path = f'Address_Failures_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    address_failures.to_csv(out_path, index=False)
    print(f"Wrote {len(address_failures):,} address failures/skips to {out_path}")


Wrote 33 address failures/skips to Address_Failures_20260729_080504.csv


## 4. Order creation summary + failures

In [6]:
if df_order_results is not None:
    print(df_order_results["status"].value_counts().to_string())
    order_failures = df_order_results[df_order_results["status"] == "failed"]
    print(f"\n{len(order_failures):,} failed orders")
else:
    order_failures = pd.DataFrame()
    print("07_Create_Subscription_Orders.ipynb has not been run yet.")

order_failures.head(20)


status
success    235
failed      21

21 failed orders


,SubscriptionUSN,TargetAccountNumber,status,plan_matched,productName,priceplanName,onebill_order_id,error
31,V113080527,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""10.346@williamsintern..."
37,V113081020,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""3.365madras@williamsi..."
62,V113085682,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""2.23allen@williamsint..."
64,V113086615,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""3.245moorhouse@willia..."
74,V113068308,SR1602,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""103.171stasaph@willia..."
78,V113083844,SR1404,failed,True,Wholesale Fibre BS2 (Chorus),Fibre 100,NaN,"Subscription identifier ""6.43porutu@williamsin..."
81,V113079289,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""3.73ferry@williamsint..."
87,V113070833,SR1404,failed,True,Wholesale Fibre BS2 (Chorus),Fibre 100,NaN,"Subscription identifier ""1.10fathom@williamsin..."
97,V113079537,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""106.20bath@williamsin..."
110,V113075618,SR1404,failed,True,Wholesale Fibre BS2 (Enable),Fibre 100,NaN,"Subscription identifier ""134.worcester@william..."


In [7]:
if not order_failures.empty:
    out_path = f'Order_Failures_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    order_failures.to_csv(out_path, index=False)
    print(f"Wrote {len(order_failures):,} order failures to {out_path}")

    error_summary = (
        order_failures.groupby("error")
        .agg(count=("SubscriptionUSN", "size"), example=("SubscriptionUSN", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
else:
    error_summary = pd.DataFrame()

error_summary


Wrote 21 order failures to Order_Failures_20260729_080504.csv


,error,count,example
0,"Subscription identifier ""1.10fathom@williamsin...",1,V113070833
1,"Subscription identifier ""1.1lava@williamsinter...",1,V113080550
2,"Subscription identifier ""10.346@williamsintern...",1,V113080527
3,"Subscription identifier ""102.162manchester@wil...",1,V113071906
4,"Subscription identifier ""102.huanuilane@willia...",1,V113066260
5,"Subscription identifier ""103.171stasaph@willia...",1,V113068308
6,"Subscription identifier ""106.20bath@williamsin...",1,V113079537
7,"Subscription identifier ""11.7ariki@williamsint...",1,V113071302
8,"Subscription identifier ""11.oliver@williamsint...",1,V113063085
9,"Subscription identifier ""12.216rosebank@willia...",1,V113064323


## 5. Orders on a fallback (unmatched) plan

Successfully created, but on `STATIC_FALLBACK_PLAN` rather than a matched
plan — worth reviewing before calling the migration done.

In [8]:
if df_order_results is not None and not df_order_results.empty:
    unmatched_plans = df_order_results[
        (df_order_results["status"] == "success") & (df_order_results["plan_matched"] == False)  # noqa: E712
    ]
    print(f"{len(unmatched_plans):,} successfully-created orders used the STATIC_FALLBACK_PLAN")
else:
    unmatched_plans = pd.DataFrame()

unmatched_plans


0 successfully-created orders used the STATIC_FALLBACK_PLAN


,SubscriptionUSN,TargetAccountNumber,status,plan_matched,productName,priceplanName,onebill_order_id,error


## 6. End-to-end funnel

In [9]:
_df_subs = try_load_df("subscriptions_resolved")
funnel = {
    "subscriptions loaded":     len(_df_subs) if _df_subs is not None else 0,
    "addresses created":        (df_address_results["status"] == "created").sum() if df_address_results is not None else 0,
    "orders attempted":         len(df_order_results) if df_order_results is not None else 0,
    "orders succeeded":         (df_order_results["status"] == "success").sum() if df_order_results is not None else 0,
}
pd.Series(funnel, name="count").to_frame()


,count
subscriptions loaded,264
addresses created,231
orders attempted,256
orders succeeded,235
